# 02 U-Net（Oxford-IIIT Pet）: スキップ接続は何を担っているのか？

ランタイムを **GPU（T4）** にしてから上から順に実行する。
1. リポジトリを clone → 2. `--quick` で動作確認 → 3. A/B を各 seed で学習 → 4. 評価・比較図 → 5. `results/` を zip でダウンロード

合計の目安: 1条件（1 seed）あたり40分以内になる設定（`--epochs 30`）。実測時間は `train_info.json` の `train_sec` を見る。

In [ ]:
# リポジトリを clone する。ブランチを main にマージしたら BRANCH を "main" に変える
REPO = "https://github.com/kimhuisong/summer-cv-practice.git"
BRANCH = "claude/jolly-bell-qm0rpb"
!git clone -b $BRANCH $REPO
%cd summer-cv-practice/02_unet_oxfordpet
!nvidia-smi | head -15

In [ ]:
# Colab には torch / torchvision / matplotlib が入っている。足りなければ入れる
!pip install -q matplotlib pillow

In [ ]:
# 動作確認（CPU・一部データ・1エポック）。ここの数値は本番の結果ではない
!python model.py
!python train.py --variant unet --quick
!python train.py --variant noskip --quick
!python evaluate.py --quick

In [ ]:
# データの取得確認（初回は torchvision が公式配布元から images.tar.gz / annotations.tar.gz をダウンロードして前処理・キャッシュ）
from data import get_datasets
tr, va, te, real = get_datasets("data", seed=0)
print(len(tr), len(va), len(te), "real_data =", real)

In [ ]:
# 本番学習: 条件A（unet）と条件B（noskip）を seed 0,1,2 で学習する
SEEDS = [0, 1, 2]
for seed in SEEDS:
    for variant in ["unet", "noskip"]:
        !python train.py --variant $variant --seed $seed --epochs 30

In [ ]:
# 評価と比較図（results/comparison/ に metrics.json と画像を保存）
!python evaluate.py --seeds 0 1 2

In [ ]:
from IPython.display import Image, display
display(Image("results/comparison/per_class_iou.png"))
display(Image("results/comparison/prediction_comparison.png"))

In [ ]:
# results/ を zip にまとめてダウンロード（チェックポイントは重いので除外）
!zip -r results.zip results -x "*/checkpoints/*" "results/quick/*"
from google.colab import files
files.download("results.zip")